# Cobblemon Atlas｜10 隻跨骨架動畫批次壓力測試

此 Notebook 將自動挑選 **10 種形態、每種 3 個不同動作**，在 Google Colab CPU 使用真實渲染器，逐 job 寫入 Google Drive。GPU 測試僅為獨立光柵化數學微基準，**不是完整 GPU WebP 渲染器**。

必須已存在 `Cobblemon_Atlas/source_packs/{merged_pack.zip,development_resource_packs.zip,render_portraits.py}` 和 `manifests/animation-actions.json`。不修改 GitHub Pages 與原有 1,170 張 WebP；錯誤和靜態動作保留於報告。

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, json, zipfile, time
from collections import Counter
DRIVE_ROOT = Path('/content/drive/MyDrive/Cobblemon_Atlas')
WORK_ROOT = Path('/content/cobblemon-bedrock')
SITE_ROOT = Path('/content/cobblemon-atlas')
BRANCH = 'feat/animation-manifest-scanner-phase1'
# 初期保守測試：10 個形態 × 3 動作，24 幀、224px，不覆寫既有 WebP。
TARGET_FORMS, ACTIONS_PER_FORM = 10, 3
FPS, SECONDS, SIZE, SS = 12, 2, 224, 2
# 種類分散的預選；不存在／缺少 3 個有效動作時，從 Manifest 補足。
FORM_PREFERENCES = [
    '0025_pikachu', '0006_charizard', '0094_gengar',
    '0130_gyarados_mega', '0149_dragonite', '0249_lugia',
    '0384_rayquaza', '0448_lucario', '0658_greninja',
    '0700_sylveon', '0003_venusaur', '0009_blastoise',
    '0150_mewtwo', '0282_gardevoir', '0890_eternatus',
]
print('Batch target:', TARGET_FORMS, 'forms ×', ACTIONS_PER_FORM, 'actions')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert DRIVE_ROOT.is_dir(), f'找不到雲端資料夾 {DRIVE_ROOT}；請確認使用正確 Google 帳號'
for name in ['source_packs','manifests','renders','reports','checkpoints']:
    assert (DRIVE_ROOT/name).is_dir(), f'缺少 Drive 資料夾：{name}'
print('Google Drive 目錄確認成功:', DRIVE_ROOT)

In [ ]:
if not (SITE_ROOT/'.git').is_dir():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse','--branch',BRANCH,'https://github.com/hwh0410/cobblemon-atlas.git',str(SITE_ROOT)],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'sparse-checkout','set','tools','docs','notebooks','mega-showdown/data'],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'pull','--ff-only','origin',BRANCH],check=True)
sha=subprocess.check_output(['git','-C',str(SITE_ROOT),'rev-parse','--short','HEAD'],text=True).strip()
renderer_source=(SITE_ROOT/'tools/render_action_jobs.py').read_text()
assert 'def choose_actions(' in renderer_source, '目前載入的是舊版選擇器，請重新開啟 GitHub 最新 Notebook'
print('已載入最新版候選來源選擇器，Git commit:',sha)

In [ ]:
# 使用本機暫存磁碟渲染，避免每一幀反覆從 Drive 讀取大量小檔案。
WORK_ROOT.mkdir(parents=True,exist_ok=True)
packs = DRIVE_ROOT/'source_packs'
required = [packs/'development_resource_packs.zip', packs/'merged_pack.zip', packs/'render_portraits.py', DRIVE_ROOT/'manifests/animation-actions.json']
missing = [str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('請先上傳以下來源檔：\n'+'\n'.join(missing))
for archive in required[:2]:
    expected = WORK_ROOT/archive.stem
    if expected.is_dir():
        print('Reuse extracted pack:', expected)
        continue
    print('Extracting:', archive.name, flush=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            dest = (WORK_ROOT/item.filename).resolve()
            if not dest.is_relative_to(WORK_ROOT.resolve()): raise ValueError('Unsafe archive path')
        z.extractall(WORK_ROOT)
    assert expected.is_dir(), f'壓縮檔最外層不是 {archive.stem}，請重新封裝'
(WORK_ROOT/'tools').mkdir(exist_ok=True)
shutil.copy2(packs/'render_portraits.py', WORK_ROOT/'tools/render_portraits.py')
manifest = DRIVE_ROOT/'manifests/animation-actions.json'
data=json.loads(manifest.read_text())
print('Manifest:',data['summary'])
print('Resources prepared locally')

In [ ]:
# 依實際 Manifest 選擇來源與骨骼相容的 10×3 job，不假設形態或 clip 存在。
import importlib.util
tool_path=SITE_ROOT/'tools/render_action_jobs.py'
spec=importlib.util.spec_from_file_location('atlas_batch_jobs',tool_path)
mod=importlib.util.module_from_spec(spec);spec.loader.exec_module(mod)
ignored_terms=('evolution','ride_','_test','quirk','blink','recoil','look','old','test')
def ranked_action_names(entry):
    unique={}
    for a in entry.get('actions',[]):
        name=a.get('clip_key','').rsplit('.',1)[-1]
        matched=int(a.get('model_bones_matched') or 0)
        moving=int(a.get('dynamic_bones') or 0)
        if not name or not name.replace('_','').isalnum() or matched<=0 or moving<=0:
            continue
        if any(term in name for term in ignored_terms):
            continue
        score=(matched/max(1,moving),matched,a.get('source_pack')=='merged_pack')
        if name not in unique or score>unique[name][0]:
            unique[name]=(score,a)
    names=list(unique)
    def category(name):
        if any(s in name for s in ('idle','standing','hover','floating','pose')): return 'idle'
        if any(s in name for s in ('walk','fly','swim','run','move')): return 'move'
        if any(s in name for s in ('battle','attack','cry')): return 'battle'
        return 'other'
    names.sort(key=lambda n:(category(n)!='idle',-unique[n][0][1],n))
    selected=[]
    for target in ('idle','move','battle'):
        matches=[n for n in names if category(n)==target and n not in selected]
        if matches: selected.append(matches[0])
    for n in names:
        if len(selected)>=ACTIONS_PER_FORM: break
        if n not in selected: selected.append(n)
    return selected[:ACTIONS_PER_FORM]
entries={e['form_id']:e for e in data['entries']}
ranked=[]
for form in [*FORM_PREFERENCES,*sorted(entries)]:
    if form in entries and form not in ranked and len(ranked_action_names(entries[form]))>=ACTIONS_PER_FORM:
        ranked.append(form)
    if len(ranked)>=TARGET_FORMS: break
assert len(ranked)==TARGET_FORMS, '動作足夠的形態不足 10 個'
batch=[]
for form in ranked:
    names=ranked_action_names(entries[form])
    choices=mod.choose_actions([entries[form]],{form},names)
    assert len(choices)==ACTIONS_PER_FORM
    for e,a,output_id in choices:
        assert a['model_bones_matched']>0 and a['dynamic_bones']>0
        # Auto selection must agree with filtered candidates.
        batch.append({'form':form,'action':output_id,'clip_key':a['clip_key'],
                      'source_pack':a['source_pack'],'matched_bones':a['model_bones_matched']})
assert len(batch)==TARGET_FORMS*ACTIONS_PER_FORM
plan={'git_commit':sha,'forms':ranked,'jobs':batch,
      'fps':FPS,'seconds':SECONDS,'size':SIZE,'ss':SS}
plan_file=DRIVE_ROOT/'reports/batch-pilot-10-plan.json'
plan_file.write_text(json.dumps(plan,ensure_ascii=False,indent=2))
print('Selected forms:',ranked)
for item in batch: print(item['form'],item['action'],item['source_pack'],item['matched_bones'])
print('Plan:',plan_file)

In [ ]:
# 逐 form 執行，避免 --forms × --actions 意外形成交叉積。
# 每個動作完成都會同步至 Drive；失敗不阻擋其他形態，重新執行可續跑。
out=WORK_ROOT/'captures/atlas_action_pilot'
start=time.perf_counter()
exit_codes={}
for ix,form in enumerate(ranked,1):
    names=[a['action'] for a in batch if a['form']==form]
    cmd=[sys.executable,str(tool_path),'--repo',str(WORK_ROOT),
         '--manifest',str(manifest),'--forms',form,'--actions',*names,
         '--fps',str(FPS),'--seconds',str(SECONDS),'--size',str(SIZE),
         '--ss',str(SS),'--auto-frame-attempts','8',
         '--out',str(out),'--sync-root',str(DRIVE_ROOT)]
    print(f'\\n[{ix}/{TARGET_FORMS}] {form} → {names}',flush=True)
    result=subprocess.run(cmd,check=False)
    exit_codes[form]=result.returncode
    print('exit:',result.returncode,'elapsed:',round(time.perf_counter()-start,1),'s',flush=True)
print('Batch finished:',round(time.perf_counter()-start,2),'seconds')

In [ ]:
# GPU 算術微基準（不包含完整動畫渲染、材質及 WebP 編碼）。
gpu_report=DRIVE_ROOT/'reports/raster-cpu-gpu-benchmark.json'
subprocess.run([sys.executable,str(SITE_ROOT/'tools/benchmark_raster_gpu.py'),
                '--triangles','32','--size',str(SIZE),'--repeats','5',
                '--output',str(gpu_report)],check=True)
print('Microbenchmark:',gpu_report)

In [ ]:
# 僅統計本輪 30 個 job，不把先前洛奇亞等已完成的工作混入。
checkpoint=DRIVE_ROOT/'checkpoints/render-jobs.json'
assert checkpoint.is_file(), '缺少 checkpoint'
job_data=json.loads(checkpoint.read_text())['jobs']
sampled={}
for item in batch:
    key=item['form']+'/'+item['action']
    sampled[key]=job_data.get(key,{'status':'missing','reason':'no_checkpoint'})
statuses=Counter(x.get('status','unknown') for x in sampled.values())
new_work=sum(float(x.get('elapsed_sec',0)) for x in sampled.values() if x.get('elapsed_sec') is not None)
summary={'git_commit':sha,'selected_forms':ranked,'total_jobs':len(batch),
         'status_counts':dict(statuses),'sum_job_elapsed_sec':round(new_work,2),
         'jobs':{k:{s:v.get(s) for s in ('status','reason','elapsed_sec','bytes',
                  'n_frames','decoded_frames','changed_pixels_max','min_margin_px',
                  'framing_attempts','source_pack','clip_key')} for k,v in sampled.items()}}
summary_file=DRIVE_ROOT/'reports/batch-pilot-10-summary.json'
summary_file.write_text(json.dumps(summary,ensure_ascii=False,indent=2))
print('Status:',dict(statuses))
print('Sum job CPU seconds (including stored previous jobs):',round(new_work,2))
for key,v in sampled.items(): print(key,v['status'],v.get('reason'))
print('Report:',summary_file)
print('Checkpoint:',checkpoint)